# Análise agregada dos processos de prompt optimization do `item_knn`

Este notebook percorre os processos finalizados em `out/prompt_optimization/Llama3.1-I/item_knn` e ajuda a responder quais configurações tiveram melhor desempenho.

O foco aqui é comparar processos completos dentro do algoritmo:
- destacar o melhor processo por métrica de objetivo usando `best_train_metric`;
- mostrar o ranking completo de configurações;
- plotar a evolução de treino e validação por época;
- indicar se já existe algum resultado de teste ligado ao `best_prompt.json`.

Observações:
- O melhor processo é definido pelo maior `best_train_metric`, que representa o melhor valor de treino atingido durante a otimização.
- Quando há empate em `best_train_metric`, o desempate usa `best_val_metric` e depois menor `time_prompt_optimization`.
- Métricas de objetivo diferentes não devem ser comparadas diretamente entre si; o notebook separa os rankings por métrica.
- Se `out/test_explainability` ainda não existir, a parte de resultados de teste aparecerá vazia sem quebrar a análise.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Não foi possível localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook no projeto, em um subdiretório dele ou a partir da raiz do workspace."
    )

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.optimization_process_analysis import (
    discover_optimization_processes,
    load_process_bundle,
    summarize_processes,
)

METRIC_COLOR_MAP = {
    "etd": "#1f77b4",
    "sep": "#ff7f0e",
    "sep_etd_f1": "#2ca02c",
    "geom_mean": "#d62728",
    "mean_balance": "#8c564b",
}


def metric_color(metric_name: str | None, default: str = "#4C78A8") -> str:
    return METRIC_COLOR_MAP.get(str(metric_name), default)


METRIC_LABEL_MAP = {
    "etd": "ETD",
    "sep": "SEP",
    "sep_etd_f1": "SEP_ETD_F1",
    "geom_mean": "Geométrica",
    "mean_balance": "Mean Balance",
}


def metric_label(metric_name: str | None) -> str:
    return METRIC_LABEL_MAP.get(str(metric_name), str(metric_name))


def blend_with_white(color: str, blend: float) -> str:
    red, green, blue = mcolors.to_rgb(color)
    return mcolors.to_hex(
        tuple((1 - blend) * channel + blend for channel in (red, green, blue))
    )


def metric_shades(metric_name: str | None, size: int) -> list[str]:
    base_color = metric_color(metric_name)
    if size <= 1:
        return [base_color]

    max_blend = 0.45
    return [
        blend_with_white(base_color, max_blend * (index / max(1, size - 1)))
        for index in range(size)
    ]


def ensure_balance_metrics(epochs_df: pd.DataFrame) -> pd.DataFrame:
    epochs_df = epochs_df.copy()

    for split in ("train", "val"):
        sep_col = f"{split}_score_sep"
        etd_col = f"{split}_score_etd"

        if sep_col not in epochs_df.columns or etd_col not in epochs_df.columns:
            continue

        sep_series = pd.to_numeric(epochs_df[sep_col], errors="coerce")
        etd_series = pd.to_numeric(epochs_df[etd_col], errors="coerce")

        epochs_df[f"{split}_score_geom_mean"] = (
            sep_series.clip(lower=0) * etd_series.clip(lower=0)
        ).pow(0.5)
        epochs_df[f"{split}_score_mean_balance"] = (
            ((sep_series + etd_series) / 2.0)
            * (1 - (sep_series - etd_series).abs())
        )

    return epochs_df


def preferred_metric_order(metric_names: list[str]) -> list[str]:
    preferred = ["sep", "etd", "sep_etd_f1", "geom_mean", "mean_balance"]
    ordered = [metric_name for metric_name in preferred if metric_name in metric_names]
    ordered.extend(metric_name for metric_name in metric_names if metric_name not in ordered)
    return ordered

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

PROJECT_ROOT


In [ ]:
ALGORITHM_NAME = "item_knn"
SEARCH_ROOT_REL = "out/prompt_optimization/Llama3.1-I/item_knn"
SEARCH_ROOT = PROJECT_ROOT / SEARCH_ROOT_REL
TEST_ROOT = PROJECT_ROOT / "out" / "test_explainability"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALGORITHM_NAME:", ALGORITHM_NAME)
print("SEARCH_ROOT:", SEARCH_ROOT)
print("TEST_ROOT_EXISTS:", TEST_ROOT.exists())


In [ ]:
process_catalog = discover_optimization_processes(PROJECT_ROOT, search_root=SEARCH_ROOT)
if process_catalog.empty:
    raise FileNotFoundError("Nenhum optimization_process_metadata.json foi encontrado em " + str(SEARCH_ROOT))

process_catalog = process_catalog.copy()
process_catalog["early_label"] = "early_" + process_catalog["early_stopping"].astype(str).str.lower()
process_catalog["process_label"] = (
    process_catalog["objective_metric"].astype(str)
    + " | "
    + process_catalog["representation_model"].astype(str)
    + " | "
    + process_catalog["early_label"].astype(str)
    + " | lambda="
    + process_catalog["mmr_lambda_quality"].astype(str)
    + " | pool="
    + process_catalog["mmr_pool_multiplier"].astype(str)
)
process_catalog["process_notebook_path_rel"] = process_catalog["process_dir_rel"] + "/plot_optimization_process.ipynb"

display(Markdown("## Catálogo dos processos encontrados em `" + SEARCH_ROOT_REL + "`"))
display(
    summarize_processes(
        process_catalog,
        columns=[
            "objective_metric",
            "representation_model",
            "early_stopping",
            "mmr_lambda_quality",
            "mmr_pool_multiplier",
            "epochs_completed",
            "best_train_metric",
            "best_val_metric",
            "saved_best_origin",
            "process_dir_rel",
        ],
    )
)
print("Quantidade de processos finalizados:", len(process_catalog))


In [ ]:
ranking_view = process_catalog.sort_values(
    by=["objective_metric", "best_train_metric", "best_val_metric", "time_prompt_optimization"],
    ascending=[True, False, False, True],
    na_position="last",
).reset_index(drop=True)

best_by_metric = (
    ranking_view.groupby("objective_metric", as_index=False)
    .first()
    .loc[
        :,
        [
            "objective_metric",
            "metric_name",
            "representation_model",
            "early_stopping",
            "mmr_lambda_quality",
            "mmr_pool_multiplier",
            "best_train_metric",
            "best_val_metric",
            "epochs_completed",
            "time_prompt_optimization",
            "process_dir_rel",
            "process_notebook_path_rel",
        ],
    ]
)

display(Markdown("## Melhor processo por métrica de objetivo"))
display(best_by_metric)
display(
    Markdown(
        "> Ranking principal: `best_train_metric` decrescente. Desempates usam `best_val_metric` e depois menor `time_prompt_optimization`."
    )
)


In [ ]:
display(Markdown("## Ranking completo por métrica"))
ranking_columns = [
    "process_label",
    "representation_model",
    "early_stopping",
    "mmr_lambda_quality",
    "mmr_pool_multiplier",
    "best_train_metric",
    "best_val_metric",
    "epochs_completed",
    "time_prompt_optimization",
    "process_notebook_path_rel",
]

for objective_metric, metric_df in ranking_view.groupby("objective_metric", sort=True):
    ordered = metric_df.reset_index(drop=True)
    display(Markdown("### " + str(objective_metric)))
    display(ordered.loc[:, ranking_columns])

    chart_df = ordered.dropna(subset=["best_train_metric"]).copy()
    if chart_df.empty:
        display(Markdown("> Nenhum `best_train_metric` disponível para plotar neste grupo."))
        continue

    chart_colors = metric_shades(objective_metric, len(chart_df))
    fig_height = max(4.0, 0.65 * len(chart_df))
    fig, ax = plt.subplots(figsize=(13, fig_height))
    ax.barh(chart_df["process_label"], chart_df["best_train_metric"], color=chart_colors)
    ax.invert_yaxis()
    ax.set_xlabel("best_train_metric")
    ax.set_ylabel("processo")
    ax.set_title("Melhores valores de treino")

    max_value = chart_df["best_train_metric"].max()
    offset = max_value * 0.01 if pd.notna(max_value) and max_value != 0 else 0.01
    for index, value in enumerate(chart_df["best_train_metric"]):
        ax.text(value + offset, index, f"{value:.4f}", va="center")

    plt.tight_layout()
    plt.show()


In [ ]:
bundles = {}
linked_rows = []

for _, row in process_catalog.iterrows():
    bundle = load_process_bundle(Path(row["process_dir"]), PROJECT_ROOT)
    bundles[row["process_dir"]] = bundle
    linked_payload = bundle["linked_test_metadata"] or {}
    linked_rows.append(
        {
            "objective_metric": row["objective_metric"],
            "process_label": row["process_label"],
            "linked_test_metadata_path": bundle["summary"]["linked_test_metadata_path"],
            "test_metric_value": linked_payload.get("metric_value"),
            "n_users": linked_payload.get("n_users"),
            "time_to_explain": linked_payload.get("time_to_explain"),
            "prompt_source": linked_payload.get("prompt_source"),
        }
    )

linked_tests_df = pd.DataFrame(linked_rows)

display(Markdown("## Ligação com resultados de teste"))
if linked_tests_df["linked_test_metadata_path"].notna().any():
    display(linked_tests_df)
else:
    display(
        Markdown(
            "> Nenhum `responses_metadata.json` ligado aos `best_prompt.json` foi encontrado em `out/test_explainability`."
        )
    )


In [ ]:
display(Markdown("## Curvas por época dos processos"))

for objective_metric, metric_df in ranking_view.groupby("objective_metric", sort=True):
    display(Markdown("### " + str(objective_metric)))
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharex=True)
    train_ax, val_ax = axes
    plotted_train = False
    plotted_val = False

    curve_colors = metric_shades(objective_metric, len(metric_df))

    for color, (_, row) in zip(curve_colors, metric_df.iterrows()):
        epochs_df = bundles[row["process_dir"]]["epochs_df"]
        if epochs_df.empty:
            continue

        label = row["process_label"]
        if "train_metric" in epochs_df and epochs_df["train_metric"].notna().any():
            train_ax.plot(
                epochs_df["epoch"],
                epochs_df["train_metric"],
                marker="o",
                linewidth=2,
                color=color,
                label=label,
            )
            plotted_train = True

        if "val_metric" in epochs_df and epochs_df["val_metric"].notna().any():
            val_ax.plot(
                epochs_df["epoch"],
                epochs_df["val_metric"],
                marker="o",
                linewidth=2,
                color=color,
                label=label,
            )
            plotted_val = True

    train_ax.set_title("Treino")
    train_ax.set_xlabel("época")
    train_ax.set_ylabel("train_metric")

    val_ax.set_title("Validação")
    val_ax.set_xlabel("época")
    val_ax.set_ylabel("val_metric")

    if plotted_train:
        train_ax.legend(loc="best", fontsize=8)
    else:
        train_ax.text(0.5, 0.5, "Sem dados de treino", ha="center", va="center", transform=train_ax.transAxes)

    if plotted_val:
        val_ax.legend(loc="best", fontsize=8)
    else:
        val_ax.text(0.5, 0.5, "Sem dados de validação", ha="center", va="center", transform=val_ax.transAxes)

    plt.tight_layout()
    plt.show()
